## CELL 1 — Config & Seed

In [1]:
"""
NOTEBOOK 1B.3 - ICAM + SupCon (lambda=0.3) + curriculum
Fase 1B Ronde 2 - IndoMeme-XAI

Arsitektur: IndoBERTweet + ViT -> ICAM -> [SupCon proj head | topic concat -> classifier]
Loss: L_focal + lambda * L_supcon (supcon aktif epoch 4+)
Tujuan: validasi training strategy SupCon. Target: Macro-F1 > 0.81.
"""

from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input/datasets/reynaldaryansyah")
ROOT_DIR     = KAGGLE_INPUT / "xai-dataset" / "data"
CSV_PATH     = ROOT_DIR / "processed" / "all_splits.csv"
IMG_DIR      = ROOT_DIR / "images"

OUTPUT_DIR   = Path("/kaggle/working/exp_1B3_icam_supcon")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = dict(
    exp_id           = "1B.3_icam_supcon_lambda03",
    seed             = 42,

    text_model_name  = "indolem/indobertweet-base-uncased",
    image_model_name = "google/vit-base-patch16-224-in21k",
    freeze_n_layers  = 4,

    topic_emb_dim    = 32,
    head_hidden      = 256,
    dropout          = 0.3,

    icam_num_heads   = 8,
    icam_dropout     = 0.1,

    # SupCon (NEW)
    supcon_lambda          = 0.3,
    supcon_temperature     = 0.07,
    supcon_proj_hidden     = 128,
    supcon_proj_out        = 128,
    supcon_warmup_epochs   = 3,        # epoch 1-3 = focal only, epoch 4+ = +supcon

    max_seq_len      = 128,

    # Training (lebih panjang krn curriculum SupCon)
    batch_size       = 32,
    grad_accum       = 2,
    num_epochs       = 20,             # naik dari 15
    early_stop_pat   = 7,              # naik dari 5
    grad_clip        = 1.0,

    lr_encoder       = 1e-5,
    lr_fusion        = 3e-5,
    lr_head          = 5e-5,
    weight_decay     = 0.01,
    warmup_ratio     = 0.10,

    focal_gamma      = 2.0,
    label_smoothing  = 0.1,
    class_weights    = [0.76, 1.46],

    thr_search_lo    = 0.30,
    thr_search_hi    = 0.70,
    thr_search_step  = 0.01,
)

print(f"Experiment: {CONFIG['exp_id']}")
print(f"Output dir: {OUTPUT_DIR}")

Experiment: 1B.3_icam_supcon_lambda03
Output dir: /kaggle/working/exp_1B3_icam_supcon


## CELL 2 — Imports & Reproducibility

In [2]:
import os, json, random, time, math, re, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler

from PIL import Image
from sklearn.metrics import (
    f1_score, precision_recall_fscore_support, accuracy_score,
    roc_auc_score, classification_report, confusion_matrix,
)
from torchvision import transforms
from transformers import (
    AutoTokenizer, AutoModel, ViTImageProcessor, ViTModel,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(CONFIG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}  |  PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda  |  PyTorch: 2.10.0+cu128
GPU: Tesla T4


## CELL 3 — Slang Dictionary

In [3]:
SLANG_DICT = {
    "gw":"saya","gue":"saya","gua":"saya","guwe":"saya",
    "lu":"kamu","lo":"kamu","loe":"kamu","elo":"kamu","elu":"kamu",
    "ane":"saya","ente":"kamu","agan":"kamu","sis":"kakak",
    "bro":"kawan","bray":"kawan","bre":"kawan","cuy":"kawan",
    "klian":"kalian","kmu":"kamu","kamuh":"kamu",
    "aq":"aku","akoh":"aku","akuh":"aku",
    "gak":"tidak","ga":"tidak","gk":"tidak","nggak":"tidak",
    "ngga":"tidak","ngk":"tidak","kgk":"tidak","tdk":"tidak",
    "engga":"tidak","enggak":"tidak","kga":"tidak",
    "yg":"yang","dgn":"dengan","dg":"dengan","krn":"karena",
    "krna":"karena","utk":"untuk","tp":"tapi","tpi":"tapi",
    "klo":"kalau","klau":"kalau","kl":"kalau","klu":"kalau",
    "spy":"supaya","ato":"atau",
    "bgt":"banget","bgtt":"banget","bngt":"banget",
    "bnr":"benar","bener":"benar",
    "emg":"memang","emang":"memang","mmg":"memang",
    "skrg":"sekarang","skrng":"sekarang",
    "td":"tadi","tdi":"tadi","bsk":"besok",
    "knp":"kenapa","gmn":"bagaimana","gimana":"bagaimana",
    "gmana":"bagaimana","kyk":"seperti","kayak":"seperti","kek":"seperti",
    "udh":"sudah","udah":"sudah","uda":"sudah","dah":"sudah",
    "blm":"belum","blom":"belum",
    "lg":"lagi","lgi":"lagi","msh":"masih","msi":"masih",
    "jd":"jadi","jdi":"jadi",
    "tau":"tahu","taw":"tahu","tw":"tahu",
    "liat":"lihat","ngeliat":"melihat","denger":"dengar",
    "ngomong":"bicara","blng":"bilang","bilg":"bilang",
    "aja":"saja","doang":"saja","doank":"saja",
    "sih":"","deh":"","dong":"","kok":"","lah":"",
    "loh":"","lho":"","yah":"",
    "iy":"iya","iyaa":"iya","iye":"iya","yup":"iya","yoi":"iya",
    "org":"orang","orng":"orang","orgnya":"orangnya",
    "dr":"dari","dri":"dari","kpd":"kepada",
    "thd":"terhadap","ttg":"tentang",
    "abis":"habis","trs":"terus","trus":"terus",
    "tmn":"teman","temen":"teman","tman":"teman",
    "rmh":"rumah","kntr":"kantor","krj":"kerja",
    "ank":"anak","indo":"indonesia",
    "cwo":"cowok","cwe":"cewek","cwok":"cowok","cwek":"cewek",
    "wkwk":"haha","wkwkwk":"haha","wkwkwkwk":"haha","wkwkwkwkwk":"haha",
    "hahaha":"haha","hahahaha":"haha",
    "hehehe":"hehe","hehehehe":"hehe",
    "anjir":"anjing","anjay":"anjing","anjrit":"anjing",
    "anjr":"anjing","anying":"anjing","njir":"anjing",
    "bgsd":"bangsat","bgst":"bangsat",
    "bjir":"anjing","bjg":"bajingan",
    "kntl":"kontol","mmk":"memek","tai":"tahi",
}
print(f"Slang dict entries: {len(SLANG_DICT)}")

Slang dict entries: 149


## CELL 4 — clean_ocr

In [4]:
URL_PATTERN     = re.compile(r"(https?://\S+|www\.\S+|\S+\.(?:com|co\.id|net|org|id|tv|me)\b\S*)", re.IGNORECASE)
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
ALLOWED_PUNCT   = re.compile(r"[^\w\s\.,!?\-'\"]+", re.UNICODE)
MULTI_SPACE     = re.compile(r"\s+")

def _normalize_unicode(t): return unicodedata.normalize("NFKC", t)

def _normalize_slang(text, slang_dict=SLANG_DICT):
    out = []
    for tok in text.split():
        prefix, suffix, core = "", "", tok
        while core and not core[0].isalnum(): prefix += core[0]; core = core[1:]
        while core and not core[-1].isalnum(): suffix = core[-1] + suffix; core = core[:-1]
        out.append(f"{prefix}{slang_dict.get(core.lower(), core)}{suffix}")
    return " ".join(out)

def clean_ocr(text):
    if text is None: return ""
    if isinstance(text, float) and pd.isna(text): return ""
    text = _normalize_unicode(str(text)).lower()
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(r"\1", text)
    text = ALLOWED_PUNCT.sub(" ", text)
    text = _normalize_slang(text)
    return MULTI_SPACE.sub(" ", text).strip()

## CELL 5 — format_input 

In [5]:
SEP_TOKEN = "[SEP]"

def _safe_str(x):
    if x is None: return ""
    if isinstance(x, float) and pd.isna(x): return ""
    return str(x).strip()

def format_input(topic, caption, ocr_cleaned, sep=SEP_TOKEN):
    if topic is None or (isinstance(topic, float) and pd.isna(topic)):
        topic_str = "unknown"
    elif isinstance(topic, (list, tuple, set)):
        items = sorted({str(t).lower().strip() for t in topic if t})
        topic_str = ",".join(items) if items else "unknown"
    else:
        topic_str = str(topic).lower().strip() or "unknown"
    return f"[{topic_str}] {_safe_str(caption)} {sep} {_safe_str(ocr_cleaned)}".strip()

## CELL 6 — Load Data

In [6]:
df = pd.read_csv(CSV_PATH)
print(f"Total samples: {len(df)}")
print(df["split"].value_counts().to_dict())

LABEL_MAP = {"not hate": 0, "hate": 1}
df["label"] = df["hate_final"].map(LABEL_MAP)
assert df["label"].notna().all()

df["ocr_cleaned"] = df["ocr"].apply(clean_ocr)
df["text_input"]  = df.apply(lambda r: format_input(r["topic"], r["caption"], r["ocr_cleaned"]), axis=1)

def resolve_image_path(row):
    raw = row.get("local_image_path") or row.get("image_path")
    if raw is None or (isinstance(raw, float) and pd.isna(raw)): return None
    fname = Path(str(raw).replace("\\","/")).name
    return str(IMG_DIR / row["split"] / fname)

df["full_image_path"] = df.apply(resolve_image_path, axis=1)
print(df[["split","label","text_input","full_image_path"]].head(2).to_string())

Total samples: 5023
{'train': 3415, 'test': 1005, 'validation': 603}
   split  label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    text_input                                                                              full_image_path
0  train      0                                                                                                                                                                                                                                                                       [social sub-groups] Gambar ini sec

## CELL 7 — Topic Multi-Hot 

In [7]:
TOPIC_LIST = [
    "political","national origin","individual","institutions",
    "gender","religion","social sub-groups","unknown",
]
TOPIC2IDX = {t: i for i, t in enumerate(TOPIC_LIST)}
N_TOPICS  = len(TOPIC_LIST)

def topic_to_multihot(topic):
    vec = np.zeros(N_TOPICS, dtype=np.float32)
    if topic is None or (isinstance(topic, float) and pd.isna(topic)):
        vec[TOPIC2IDX["unknown"]] = 1.0; return vec
    parts = [p.strip().lower() for p in str(topic).split(",") if p.strip()]
    found = False
    for p in parts:
        if p in TOPIC2IDX:
            vec[TOPIC2IDX[p]] = 1.0; found = True
    if not found: vec[TOPIC2IDX["unknown"]] = 1.0
    return vec

df["topic_multihot"] = df["topic"].apply(topic_to_multihot)
print(f"Topic encoding ready, N_TOPICS={N_TOPICS}")

Topic encoding ready, N_TOPICS=8


## CELL 8 — Tokenizer & Image Processor

In [8]:
tokenizer       = AutoTokenizer.from_pretrained(CONFIG["text_model_name"])
image_processor = ViTImageProcessor.from_pretrained(CONFIG["image_model_name"])

IMG_MEAN = image_processor.image_mean
IMG_STD  = image_processor.image_std
IMG_SIZE = image_processor.size["height"] if isinstance(image_processor.size, dict) else 224
print(f"Image: size={IMG_SIZE}, mean={IMG_MEAN}, std={IMG_STD}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Image: size=224, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]


## CELL 9 — Image Transforms

In [9]:
TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomPerspective(distortion_scale=0.1, p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
])
EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
])

## CELL 10 — Dataset Class 

In [10]:
class IndoMemeDataset(Dataset):
    def __init__(self, df_split, tokenizer, transform, max_seq_len=128):
        self.df          = df_split.reset_index(drop=True)
        self.tokenizer   = tokenizer
        self.transform   = transform
        self.max_seq_len = max_seq_len

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["full_image_path"]).convert("RGB")
        img_tensor = self.transform(img)
        enc = self.tokenizer(row["text_input"], padding="max_length", truncation=True,
                             max_length=self.max_seq_len, return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "pixel_values":   img_tensor,
            "topic":          torch.from_numpy(row["topic_multihot"]),
            "label":          torch.tensor(int(row["label"]), dtype=torch.long),
            "idx":            int(row["idx"]),
        }

## CELL 11 — Build Datasets

In [11]:
df_train = df[df["split"] == "train"].copy()
df_val   = df[df["split"] == "validation"].copy() if "validation" in df["split"].unique() else df[df["split"] == "val"].copy()
df_test  = df[df["split"] == "test"].copy()

ds_train = IndoMemeDataset(df_train, tokenizer, TRAIN_TRANSFORM, CONFIG["max_seq_len"])
ds_val   = IndoMemeDataset(df_val,   tokenizer, EVAL_TRANSFORM,  CONFIG["max_seq_len"])
ds_test  = IndoMemeDataset(df_test,  tokenizer, EVAL_TRANSFORM,  CONFIG["max_seq_len"])

print(f"Train: {len(ds_train)}  Val: {len(ds_val)}  Test: {len(ds_test)}")

Train: 3415  Val: 603  Test: 1005


## CELL 12 — Sampler + DataLoaders

In [12]:
labels_train = df_train["label"].values
class_count  = np.bincount(labels_train, minlength=2)
sample_w     = 1.0 / class_count[labels_train]
sampler      = WeightedRandomSampler(
    weights=torch.from_numpy(sample_w).double(),
    num_samples=len(sample_w), replacement=True,
)

dl_train = DataLoader(ds_train, batch_size=CONFIG["batch_size"], sampler=sampler,
                      num_workers=2, pin_memory=True)
dl_val   = DataLoader(ds_val,   batch_size=CONFIG["batch_size"], shuffle=False,
                      num_workers=2, pin_memory=True)
dl_test  = DataLoader(ds_test,  batch_size=CONFIG["batch_size"], shuffle=False,
                      num_workers=2, pin_memory=True)

print(f"Class count train: {class_count.tolist()}")
print(f"Steps/epoch: {len(dl_train)}  effective batch={CONFIG['batch_size']*CONFIG['grad_accum']}")

Class count train: [2242, 1173]
Steps/epoch: 107  effective batch=64


## CELL 13 — ICAM Three-Stream Fusion

In [13]:
class ICAMFusion(nn.Module):
    """
    Three-Stream Fusion (ICAM) - identik 1B.2.
    """
    def __init__(self, hidden_dim=768, num_heads=8, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.t2i_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.i2t_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.gate_relevant    = nn.Linear(2 * hidden_dim, hidden_dim)
        self.gate_incongruent = nn.Linear(2 * hidden_dim, hidden_dim)
        self.fusion_proj = nn.Linear(hidden_dim, hidden_dim)
        self.layer_norm  = nn.LayerNorm(hidden_dim)
        self.save_attn = False
        self.last_t2i_attn = self.last_i2t_attn = None
        self.last_incongruity = self.last_g1 = self.last_g2 = None

    def forward(self, text_h, image_h, text_mask=None):
        text_kpm = (~text_mask.bool()) if text_mask is not None else None

        t2i_out, t2i_w = self.t2i_attn(
            query=text_h, key=image_h, value=image_h,
            key_padding_mask=None,
            need_weights=self.save_attn, average_attn_weights=True,
        )
        i2t_out, i2t_w = self.i2t_attn(
            query=image_h, key=text_h, value=text_h,
            key_padding_mask=text_kpm,
            need_weights=self.save_attn, average_attn_weights=True,
        )
        h_image_pooled = i2t_out.mean(dim=1)

        text_norm  = F.normalize(text_h,  dim=-1)
        image_norm = F.normalize(image_h, dim=-1)
        sim         = torch.bmm(text_norm, image_norm.transpose(1, 2))
        incongruity = 1.0 - sim.abs()
        inc_w       = F.softmax(incongruity, dim=-1)
        h_inc       = torch.bmm(inc_w, image_h)

        if text_mask is not None:
            m = text_mask.unsqueeze(-1).float()
            h_inc_pooled = (h_inc * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)
        else:
            h_inc_pooled = h_inc.mean(dim=1)

        h_text = text_h[:, 0]

        g1 = torch.sigmoid(self.gate_relevant(   torch.cat([h_text, h_image_pooled], dim=-1)))
        g2 = torch.sigmoid(self.gate_incongruent(torch.cat([h_text, h_inc_pooled  ], dim=-1)))

        fused   = h_text + g1 * h_image_pooled + g2 * h_inc_pooled
        h_fused = self.layer_norm(self.fusion_proj(fused))

        if self.save_attn:
            self.last_t2i_attn    = t2i_w.detach() if t2i_w is not None else None
            self.last_i2t_attn    = i2t_w.detach() if i2t_w is not None else None
            self.last_incongruity = incongruity.detach()
            self.last_g1 = g1.detach(); self.last_g2 = g2.detach()

        return h_fused, g1.mean().item(), g2.mean().item()

## CELL 14 — Full Model dengan SupCon Projection Head

In [14]:
class ICAMHateModelSupCon(nn.Module):
    """
    Sama dgn 1B.2, tapi tambah SupCon projection head dari h_fused:
      h_fused (768) -> Linear(768, 128) -> ReLU -> Linear(128, 128) -> L2 normalize
    Output supcon_features ini dipakai L_supcon di train loop.
    h_fused asli tetap masuk ke topic concat -> classifier (tidak diubah).
    """
    def __init__(self, cfg):
        super().__init__()
        self.text_encoder  = AutoModel.from_pretrained(cfg["text_model_name"])
        self.image_encoder = ViTModel.from_pretrained(cfg["image_model_name"])

        self._freeze_bottom(self.text_encoder.encoder.layer,  cfg["freeze_n_layers"])
        self._freeze_bottom(self.image_encoder.encoder.layer, cfg["freeze_n_layers"])
        for p in self.text_encoder.embeddings.parameters():  p.requires_grad = False
        for p in self.image_encoder.embeddings.parameters(): p.requires_grad = False

        hidden = self.text_encoder.config.hidden_size
        assert hidden == self.image_encoder.config.hidden_size

        self.icam = ICAMFusion(
            hidden_dim=hidden,
            num_heads=cfg["icam_num_heads"],
            dropout=cfg["icam_dropout"],
        )

        # SupCon projection head (NEW)
        self.supcon_proj = nn.Sequential(
            nn.Linear(hidden, cfg["supcon_proj_hidden"]),
            nn.ReLU(),
            nn.Linear(cfg["supcon_proj_hidden"], cfg["supcon_proj_out"]),
        )

        self.topic_proj = nn.Sequential(
            nn.Linear(N_TOPICS, cfg["topic_emb_dim"]),
            nn.GELU(),
        )

        head_in = hidden + cfg["topic_emb_dim"]   # 768 + 32 = 800
        self.head = nn.Sequential(
            nn.Linear(head_in, cfg["head_hidden"]),
            nn.GELU(),
            nn.Dropout(cfg["dropout"]),
            nn.Linear(cfg["head_hidden"], 2),
        )

    @staticmethod
    def _freeze_bottom(layers, n):
        for i, layer in enumerate(layers):
            if i < n:
                for p in layer.parameters(): p.requires_grad = False

    def forward(self, input_ids, attention_mask, pixel_values, topic):
        t_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        i_out = self.image_encoder(pixel_values=pixel_values)

        h_fused, g1m, g2m = self.icam(
            t_out.last_hidden_state, i_out.last_hidden_state, attention_mask,
        )

        # SupCon projection (L2-normalized)
        supcon_features = F.normalize(self.supcon_proj(h_fused), dim=-1)

        # Classification path
        topic_emb = self.topic_proj(topic)
        logits    = self.head(torch.cat([h_fused, topic_emb], dim=-1))

        return logits, supcon_features, g1m, g2m


model = ICAMHateModelSupCon(CONFIG).to(DEVICE)
n_total     = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {n_total/1e6:.2f}M")
print(f"Trainable params: {n_trainable/1e6:.2f}M ({n_trainable/n_total*100:.1f}%)")

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Total params:     204.95M
Trainable params: 122.59M (59.8%)


## CELL 15 — Focal Loss 

In [15]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0, num_classes=2):
        super().__init__()
        self.gamma, self.label_smoothing, self.num_classes = gamma, label_smoothing, num_classes
        if alpha is not None:
            self.register_buffer("alpha", torch.tensor(alpha, dtype=torch.float32))
        else:
            self.alpha = None

    def forward(self, logits, target):
        log_probs = F.log_softmax(logits, dim=-1)
        probs     = log_probs.exp()
        with torch.no_grad():
            true_dist = torch.full_like(log_probs, self.label_smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.label_smoothing)
        pt      = probs.gather(1, target.unsqueeze(1)).squeeze(1).clamp(min=1e-8)
        focal_w = (1.0 - pt) ** self.gamma
        ce      = -(true_dist * log_probs).sum(dim=-1)
        loss    = focal_w * ce
        if self.alpha is not None:
            loss = self.alpha.to(logits.device)[target] * loss
        return loss.mean()

focal_criterion = FocalLoss(
    gamma=CONFIG["focal_gamma"], alpha=CONFIG["class_weights"],
    label_smoothing=CONFIG["label_smoothing"],
)

## CELL 16 — Supervised Contrastive Loss

In [16]:
class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss (Khosla et al. 2020), hate-aware:
      L = sum_i  -1/|P(i)|  *  sum_{p in P(i)}  log( exp(z_i.z_p / tau)
                                                    / sum_{a in A(i)} exp(z_i.z_a / tau) )
    P(i) = sampel di batch dengan label sama (selain i itu sendiri)
    A(i) = semua sampel di batch selain i
    Fitur z = L2-normalized projection dari h_fused.
    Label = hate (binary 0/1) -> supervisi hate-aware.
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        # features: [B, D] (sudah L2-normalized)
        # labels:   [B]
        device = features.device
        B = features.size(0)

        # Similarity matrix [B, B]
        sim = torch.matmul(features, features.T) / self.temperature

        # Stabilitas numerik: subtract max per baris
        sim_max, _ = sim.max(dim=1, keepdim=True)
        sim = sim - sim_max.detach()

        labels = labels.contiguous().view(-1, 1)
        mask_pos = (labels == labels.T).float().to(device)            # [B, B] same-class = 1
        eye      = torch.eye(B, device=device)
        mask_pos = mask_pos - eye                                     # exclude self
        mask_all = 1.0 - eye                                          # exclude self only

        exp_sim   = torch.exp(sim) * mask_all
        log_prob  = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-12)

        n_pos = mask_pos.sum(dim=1)                                   # [B]
        # Beberapa sampel mungkin tidak punya positives di batch (edge case): mask out
        mean_log_prob_pos = (mask_pos * log_prob).sum(dim=1) / n_pos.clamp(min=1.0)
        valid = (n_pos > 0).float()
        loss  = -(mean_log_prob_pos * valid).sum() / valid.sum().clamp(min=1.0)
        return loss

supcon_criterion = SupConLoss(temperature=CONFIG["supcon_temperature"])

## CELL 17 — Optimizer + Scheduler

In [17]:
def build_optimizer_scheduler(model, cfg, num_training_steps):
    """
    3 grup parameter:
      - encoder (text + image): lr_encoder = 1e-5
      - fusion (icam): lr_fusion = 3e-5
      - head (topic_proj + supcon_proj + head): lr_head = 5e-5
    """
    enc_p, fusion_p, head_p = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if n.startswith("text_encoder") or n.startswith("image_encoder"):
            enc_p.append(p)
        elif n.startswith("icam"):
            fusion_p.append(p)
        else:
            # topic_proj, supcon_proj, head
            head_p.append(p)

    groups = [
        {"params": enc_p,    "lr": cfg["lr_encoder"], "weight_decay": cfg["weight_decay"]},
        {"params": fusion_p, "lr": cfg["lr_fusion"],  "weight_decay": cfg["weight_decay"]},
        {"params": head_p,   "lr": cfg["lr_head"],    "weight_decay": cfg["weight_decay"]},
    ]
    optimizer = torch.optim.AdamW(groups)

    num_warmup = int(cfg["warmup_ratio"] * num_training_steps)
    scheduler  = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=num_warmup, num_training_steps=num_training_steps,
    )
    print(f"Param groups: encoder={sum(p.numel() for p in enc_p)/1e6:.2f}M  "
          f"fusion={sum(p.numel() for p in fusion_p)/1e6:.2f}M  "
          f"head+proj={sum(p.numel() for p in head_p)/1e6:.2f}M")
    print(f"Steps total={num_training_steps}, warmup={num_warmup}")
    return optimizer, scheduler

## CELL 18 — Metrics Utilities

In [18]:
def compute_metrics(y_true, y_pred, y_prob_hate=None):
    out = {}
    out["macro_f1"]    = f1_score(y_true, y_pred, average="macro")
    out["weighted_f1"] = f1_score(y_true, y_pred, average="weighted")
    out["accuracy"]    = accuracy_score(y_true, y_pred)
    p, r, f, _         = precision_recall_fscore_support(y_true, y_pred, labels=[0,1], zero_division=0)
    out["nothate_p"], out["nothate_r"], out["nothate_f1"] = p[0], r[0], f[0]
    out["hate_p"],    out["hate_r"],    out["hate_f1"]    = p[1], r[1], f[1]
    if y_prob_hate is not None and len(set(y_true)) > 1:
        out["roc_auc"] = roc_auc_score(y_true, y_prob_hate)
    return out

def search_threshold(y_true, y_prob_hate, lo, hi, step):
    best = (-1, 0.5)
    for thr in tqdm(np.arange(lo, hi + 1e-9, step), desc="threshold search", leave=False):
        f = f1_score(y_true, (y_prob_hate >= thr).astype(int), average="macro")
        if f > best[0]:
            best = (f, float(thr))
    return best

## CELL 19 — Train / Eval Functions dengan Curriculum 

In [19]:
@torch.no_grad()
def evaluate(model, loader, threshold=0.5, desc="eval"):
    model.eval()
    all_logits, all_labels = [], []
    for batch in tqdm(loader, desc=desc, leave=False):
        input_ids      = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        pixel_values   = batch["pixel_values"].to(DEVICE, non_blocking=True)
        topic          = batch["topic"].to(DEVICE, non_blocking=True)
        with autocast(device_type="cuda"):
            logits, _, _, _ = model(input_ids, attention_mask, pixel_values, topic)
        all_logits.append(logits.float().cpu())
        all_labels.append(batch["label"])
    logits    = torch.cat(all_logits)
    labels    = torch.cat(all_labels).numpy()
    probs     = F.softmax(logits, dim=-1).numpy()
    prob_hate = probs[:, 1]
    preds     = (prob_hate >= threshold).astype(int)
    metrics   = compute_metrics(labels, preds, prob_hate)
    metrics["threshold"] = threshold
    return metrics, prob_hate, labels


def train_one_epoch(model, loader, optimizer, scheduler,
                    focal_fn, supcon_fn, scaler, cfg, epoch):
    model.train()
    total_loss = total_focal = total_supcon = 0.0
    n = 0
    g1_log, g2_log = [], []
    supcon_active = epoch > cfg["supcon_warmup_epochs"]

    optimizer.zero_grad()
    t0   = time.time()
    pbar = tqdm(loader,
                desc=f"epoch {epoch:02d} {'F+S' if supcon_active else 'F  '}",
                leave=False)

    for step, batch in enumerate(pbar):
        input_ids      = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        pixel_values   = batch["pixel_values"].to(DEVICE, non_blocking=True)
        topic          = batch["topic"].to(DEVICE, non_blocking=True)
        labels         = batch["label"].to(DEVICE, non_blocking=True)

        with autocast(device_type="cuda"):
            logits, supcon_feats, g1m, g2m = model(input_ids, attention_mask, pixel_values, topic)
            L_focal = focal_fn(logits, labels)
            if supcon_active:
                L_supcon = supcon_fn(supcon_feats, labels)
                loss_total = L_focal + cfg["supcon_lambda"] * L_supcon
            else:
                L_supcon   = torch.tensor(0.0, device=DEVICE)
                loss_total = L_focal
            loss = loss_total / cfg["grad_accum"]

        scaler.scale(loss).backward()
        bs = labels.size(0)
        total_loss   += loss.item() * cfg["grad_accum"] * bs
        total_focal  += L_focal.item() * bs
        total_supcon += L_supcon.item() * bs
        n += bs
        g1_log.append(g1m); g2_log.append(g2m)

        if (step + 1) % cfg["grad_accum"] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], cfg["grad_clip"]
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        pbar.set_postfix(
            loss=f"{loss.item()*cfg['grad_accum']:.4f}",
            focal=f"{L_focal.item():.3f}",
            supcon=f"{L_supcon.item():.3f}",
            g1=f"{g1m:.2f}", g2=f"{g2m:.2f}",
        )

    avg_loss   = total_loss   / max(n, 1)
    avg_focal  = total_focal  / max(n, 1)
    avg_supcon = total_supcon / max(n, 1)
    print(f"  [epoch {epoch:02d}] {'F+S' if supcon_active else 'F  '}  "
          f"loss={avg_loss:.4f}  focal={avg_focal:.4f}  supcon={avg_supcon:.4f}  "
          f"g1={np.mean(g1_log):.3f}  g2={np.mean(g2_log):.3f}  "
          f"({time.time()-t0:.1f}s)")
    return avg_loss, avg_focal, avg_supcon, float(np.mean(g1_log)), float(np.mean(g2_log)), supcon_active

## CELL 20 — Main Training Loop

In [20]:
num_training_steps = math.ceil(len(dl_train) / CONFIG["grad_accum"]) * CONFIG["num_epochs"]
optimizer, scheduler = build_optimizer_scheduler(model, CONFIG, num_training_steps)
scaler = GradScaler("cuda")

best_val_f1, best_epoch, no_improve = -1.0, 0, 0
metrics_log = []
best_ckpt_path = OUTPUT_DIR / "model_best.pt"

print(f"\n{'='*72}\nTRAINING START - {CONFIG['exp_id']}")
print(f"Curriculum: epoch 1-{CONFIG['supcon_warmup_epochs']} = focal only, "
      f"epoch {CONFIG['supcon_warmup_epochs']+1}+ = focal + {CONFIG['supcon_lambda']}*supcon")
print(f"{'='*72}")

for epoch in range(1, CONFIG["num_epochs"] + 1):
    train_loss, focal_loss, supcon_loss, g1_mean, g2_mean, supcon_active = train_one_epoch(
        model, dl_train, optimizer, scheduler,
        focal_criterion, supcon_criterion, scaler, CONFIG, epoch,
    )

    val_metrics, _, _ = evaluate(model, dl_val, threshold=0.5, desc=f"epoch {epoch:02d} val")

    log_row = {
        "epoch":         epoch,
        "supcon_active": int(supcon_active),
        "train_loss":    train_loss,
        "focal_loss":    focal_loss,
        "supcon_loss":   supcon_loss,
        "g1_mean":       g1_mean,
        "g2_mean":       g2_mean,
        "val_macro_f1":  val_metrics["macro_f1"],
        "val_acc":       val_metrics["accuracy"],
        "val_hate_p":    val_metrics["hate_p"],
        "val_hate_r":    val_metrics["hate_r"],
        "val_hate_f1":   val_metrics["hate_f1"],
        "val_auc":       val_metrics.get("roc_auc", float("nan")),
    }
    metrics_log.append(log_row)
    print(f"            val: macro_f1={val_metrics['macro_f1']:.4f}  "
          f"hate_p={val_metrics['hate_p']:.3f}  hate_r={val_metrics['hate_r']:.3f}  "
          f"hate_f1={val_metrics['hate_f1']:.3f}  auc={val_metrics.get('roc_auc', float('nan')):.3f}")

    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1, best_epoch, no_improve = val_metrics["macro_f1"], epoch, 0
        torch.save({"epoch": epoch, "state_dict": model.state_dict()}, best_ckpt_path)
        with open(OUTPUT_DIR / "best_val_metrics.json", "w") as f:
            json.dump({k: float(v) for k, v in val_metrics.items()}, f, indent=2)
        print(f"            >> best_val_macro_f1={best_val_f1:.4f}, checkpoint saved")
    else:
        no_improve += 1
        if no_improve >= CONFIG["early_stop_pat"]:
            print(f"            !! early stopping (no improve {no_improve} epoch)")
            break

print(f"\nBEST val macro_f1 = {best_val_f1:.4f} at epoch {best_epoch}")
pd.DataFrame(metrics_log).to_csv(OUTPUT_DIR / "metrics_per_epoch.csv", index=False)

Param groups: encoder=114.59M  fusion=7.68M  head+proj=0.32M
Steps total=1080, warmup=108

TRAINING START - 1B.3_icam_supcon_lambda03
Curriculum: epoch 1-3 = focal only, epoch 4+ = focal + 0.3*supcon


epoch 01 F  :   0%|          | 0/107 [00:00<?, ?it/s]

  [epoch 01] F    loss=0.1802  focal=0.1802  supcon=0.0000  g1=0.500  g2=0.498  (52.2s)


epoch 01 val:   0%|          | 0/19 [00:00<?, ?it/s]

            val: macro_f1=0.7047  hate_p=0.578  hate_r=0.732  hate_f1=0.646  auc=0.807
            >> best_val_macro_f1=0.7047, checkpoint saved


epoch 02 F  :   0%|          | 0/107 [00:00<?, ?it/s]

... (58 outputs left out)

epoch 15 F+S:   0%|          | 0/107 [00:00<?, ?it/s]

  [epoch 15] F+S  loss=0.8626  focal=0.0144  supcon=2.8275  g1=0.538  g2=0.584  (44.4s)


epoch 15 val:   0%|          | 0/19 [00:00<?, ?it/s]

            val: macro_f1=0.7854  hate_p=0.725  hate_r=0.718  hate_f1=0.722  auc=0.884
            !! early stopping (no improve 7 epoch)

BEST val macro_f1 = 0.7955 at epoch 8


## CELL 21 — Threshold Calibration

In [21]:
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])

val_metrics_05, prob_hate_val, y_val = evaluate(model, dl_val, threshold=0.5, desc="val @0.50")
best_f1, best_thr = search_threshold(
    y_val, prob_hate_val,
    CONFIG["thr_search_lo"], CONFIG["thr_search_hi"], CONFIG["thr_search_step"],
)
print(f"Threshold @ 0.50 -> macro_f1={val_metrics_05['macro_f1']:.4f}")
print(f"Best threshold   -> thr={best_thr:.2f}  macro_f1={best_f1:.4f}")

val @0.50:   0%|          | 0/19 [00:00<?, ?it/s]

threshold search:   0%|          | 0/41 [00:00<?, ?it/s]

Threshold @ 0.50 -> macro_f1=0.7955
Best threshold   -> thr=0.57  macro_f1=0.7964


## CELL 22 — Final Test Evaluation 

In [22]:
test_metrics, prob_hate_test, y_test = evaluate(model, dl_test, threshold=best_thr, desc="test")
y_pred_test = (prob_hate_test >= best_thr).astype(int)

print(f"\n{'='*72}\nTEST SET RESULTS - threshold={best_thr:.2f}\n{'='*72}")
for k in ["macro_f1","weighted_f1","accuracy","hate_p","hate_r","hate_f1","roc_auc"]:
    if k in test_metrics:
        print(f"  {k:14s}: {test_metrics[k]:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred_test, target_names=["not_hate","hate"], digits=4))
print("Confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(y_test, y_pred_test))

test:   0%|          | 0/32 [00:00<?, ?it/s]


TEST SET RESULTS - threshold=0.57
  macro_f1      : 0.7999
  weighted_f1   : 0.8169
  accuracy      : 0.8149
  hate_p        : 0.7102
  hate_r        : 0.7839
  hate_f1       : 0.7452
  roc_auc       : 0.8840

Classification report:
              precision    recall  f1-score   support

    not_hate     0.8794    0.8313    0.8547       658
        hate     0.7102    0.7839    0.7452       347

    accuracy                         0.8149      1005
   macro avg     0.7948    0.8076    0.7999      1005
weighted avg     0.8210    0.8149    0.8169      1005

Confusion matrix [rows=true, cols=pred]:
[[547 111]
 [ 75 272]]


## CELL 23 — Train-Test Gap

In [23]:
ds_train_eval = IndoMemeDataset(df_train, tokenizer, EVAL_TRANSFORM, CONFIG["max_seq_len"])
dl_train_eval = DataLoader(ds_train_eval, batch_size=CONFIG["batch_size"], shuffle=False,
                           num_workers=2, pin_memory=True)

train_metrics, _, _ = evaluate(model, dl_train_eval, threshold=best_thr, desc="train-eval")
gap = train_metrics["macro_f1"] - test_metrics["macro_f1"]
print(f"Train macro_f1: {train_metrics['macro_f1']:.4f}")
print(f"Test  macro_f1: {test_metrics['macro_f1']:.4f}")
print(f"Gap (train-test): {gap:.4f}")
if gap > 0.20:
    print("WARNING: gap > 0.20 -> overfitting signal, perlu regularisasi tambahan")

train-eval:   0%|          | 0/107 [00:00<?, ?it/s]

Train macro_f1: 0.9265
Test  macro_f1: 0.7999
Gap (train-test): 0.1265


## CELL 24 — Save Final Artifacts + Summary

In [24]:
final_results = {
    "exp_id":         CONFIG["exp_id"],
    "best_epoch":     best_epoch,
    "best_threshold": best_thr,
    "supcon_active_from_epoch": CONFIG["supcon_warmup_epochs"] + 1,
    "supcon_lambda":  CONFIG["supcon_lambda"],
    "val_metrics":    {k: float(v) for k, v in val_metrics_05.items()},
    "val_metrics_at_best_thr": {"macro_f1": float(best_f1)},
    "test_metrics":   {k: float(v) for k, v in test_metrics.items()},
    "train_metrics_at_best_thr": {k: float(v) for k, v in train_metrics.items()},
    "gap_train_test": float(gap),
    "gate_means_last_epoch": {
        "g1_mean": metrics_log[-1]["g1_mean"],
        "g2_mean": metrics_log[-1]["g2_mean"],
    },
}

with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)
with open(OUTPUT_DIR / "final_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

print(f"\n{'='*72}\nSUMMARY - KIRIMKAN BLOK INI KE CHAT UNTUK UPDATE CHECKLIST\n{'='*72}")
print(f"Experiment    : {CONFIG['exp_id']}")
print(f"Best epoch    : {best_epoch}  (supcon active from epoch {CONFIG['supcon_warmup_epochs']+1})")
print(f"Threshold     : {best_thr:.2f}")
print(f"Macro-F1      : {test_metrics['macro_f1']:.4f}")
print(f"Hate P        : {test_metrics['hate_p']:.4f}")
print(f"Hate R        : {test_metrics['hate_r']:.4f}")
print(f"Hate F1       : {test_metrics['hate_f1']:.4f}")
print(f"Weighted-F1   : {test_metrics['weighted_f1']:.4f}")
print(f"Accuracy      : {test_metrics['accuracy']:.4f}")
print(f"ROC-AUC       : {test_metrics.get('roc_auc', float('nan')):.4f}")
print(f"Gap train-test: {gap:.4f}")
print(f"Gate g1 (image)        : {final_results['gate_means_last_epoch']['g1_mean']:.4f}")
print(f"Gate g2 (incongruity)  : {final_results['gate_means_last_epoch']['g2_mean']:.4f}")
print(f"\nArtifacts saved to: {OUTPUT_DIR}")


SUMMARY - KIRIMKAN BLOK INI KE CHAT UNTUK UPDATE CHECKLIST
Experiment    : 1B.3_icam_supcon_lambda03
Best epoch    : 8  (supcon active from epoch 4)
Threshold     : 0.57
Macro-F1      : 0.7999
Hate P        : 0.7102
Hate R        : 0.7839
Hate F1       : 0.7452
Weighted-F1   : 0.8169
Accuracy      : 0.8149
ROC-AUC       : 0.8840
Gap train-test: 0.1265
Gate g1 (image)        : 0.5382
Gate g2 (incongruity)  : 0.5836

Artifacts saved to: /kaggle/working/exp_1B3_icam_supcon
